# Evaluation harness — Recall@5 / MRR for the RAG retriever

The measuring stick for every comparison the professor wants. Given a
**config** (a chunk set + its embeddings + an embedding model), it runs a fixed
set of gold German questions and reports **Recall@5** and **MRR**. Swap one
component, re-run, record the number — that's how you prove "which won".

**Relevance is judged by keyword rule** (`must_contain` + `any_of` in
`gold_questions.json`), not by chunk id — so the *same* gold set scores *any*
chunking, which is what lets you compare chunking strategies fairly.

- **Recall@5** — fraction of questions where a relevant chunk is in the top 5.
- **MRR** — mean of 1/(rank of first relevant chunk); rewards ranking it higher.

> The gold set is a **DRAFT starter** — the team should verify each question is
> actually answerable from the corpus and refine the keyword rules. Garbage gold
> = meaningless scores.

Run this in the environment that has `sentence-transformers`.


## Config — the comparison registry

In [ ]:
import json
from pathlib import Path
import numpy as np

BASE = Path.cwd()
for cand in [BASE, *BASE.parents]:
    if (cand / "vector_store" / "chunks.jsonl").exists():
        BASE = cand
        break

GOLD_PATH = BASE / "eval" / "gold_questions.json"
K = 5

# Each config = one point in the comparison. `emb` may not exist yet; configs
# whose embeddings are missing are skipped with a note (build them first).
CONFIGS = [
    {
        "name": "Corpus 2 · E5-base",
        "chunks": "corpus/corpus_v2/corpus_v2_chunks.jsonl",
        "emb": "corpus/corpus_v2/embeddings_v2_e5base.npy",
        "model": "intfloat/multilingual-e5-base",
        "query_prefix": "query: ",
    },
    # --- add more rows, then run the embedding helper at the bottom to create
    #     their .npy files, then re-run this notebook ------------------------
    {
        "name": "v2 (sentence-aware) · E5-base",
        "chunks": "vector_store/chunks_v2.jsonl",
        "emb": "vector_store/embeddings_v2_e5.npy",
        "model": "intfloat/multilingual-e5-base",
        "query_prefix": "query: ",
    },
    {
        "name": "v1 (250-word) · BGE-m3",
        "chunks": "corpus/corpus_v2/corpus_v2_chunks.jsonl",
        "emb": "vector_store/embeddings_v1_bge.npy",
        "model": "BAAI/bge-m3",
        "query_prefix": "",   # bge-m3 needs NO prefix
    },
]

gold = json.loads(GOLD_PATH.read_text(encoding="utf-8"))
print(f"Project root: {BASE}")
print(f"Loaded {len(gold)} gold questions | comparing {len(CONFIGS)} configs at k={K}")


## Relevance rule + scorer (config-agnostic)

In [2]:
def is_relevant(text: str, g: dict) -> bool:
    """Keyword relevance: all must_contain present AND (no any_of OR one present)."""
    t = text.lower()
    if not all(term.lower() in t for term in g.get("must_contain", [])):
        return False
    any_of = g.get("any_of", [])
    if any_of and not any(term.lower() in t for term in any_of):
        return False
    return True


def score_config(chunks, embeddings, encode_query, gold, k=5, depth=20):
    """Return metrics + per-question detail for one config."""
    texts = [c["text"] for c in chunks]
    hits, rr, rows = 0, 0.0, []
    for g in gold:
        q = encode_query(g["question"])
        order = np.argsort(-(embeddings @ q))[:depth]
        rank = None
        for pos, idx in enumerate(order, 1):
            if is_relevant(texts[idx], g):
                rank = pos
                break
        hit = rank is not None and rank <= k
        hits += int(hit)
        rr += (1.0 / rank) if rank else 0.0
        rows.append({"id": g["id"], "hit@%d" % k: hit, "first_rank": rank})
    n = len(gold)
    return {"recall@%d" % k: hits / n, "mrr": rr / n, "rows": rows}


## Run the comparison

Loads each config's model + embeddings and scores the gold set. Configs whose
`.npy` doesn't exist yet are skipped (build them in the last cell).


In [3]:
from functools import lru_cache

@lru_cache(maxsize=None)
def get_model(name):
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer(name)


def load_chunks(path):
    return [json.loads(l) for l in (BASE / path).read_text(encoding="utf-8").splitlines()]


results = []
per_config_rows = {}
for cfg in CONFIGS:
    emb_path = BASE / cfg["emb"]
    if not emb_path.exists():
        print(f"skip  {cfg['name']:32}  (missing {cfg['emb']} — build it first)")
        continue
    chunks = load_chunks(cfg["chunks"])
    embeddings = np.load(emb_path)
    if len(chunks) != embeddings.shape[0]:
        print(f"skip  {cfg['name']:32}  (chunks {len(chunks)} != emb {embeddings.shape[0]})")
        continue
    model = get_model(cfg["model"])
    prefix = cfg["query_prefix"]
    encode_query = lambda q, m=model, p=prefix: m.encode(p + q, normalize_embeddings=True).astype("float32")
    r = score_config(chunks, embeddings, encode_query, gold, k=K)
    results.append({"config": cfg["name"], "recall@%d" % K: r["recall@%d" % K], "mrr": r["mrr"], "n_chunks": len(chunks)})
    per_config_rows[cfg["name"]] = r["rows"]
    print(f"done  {cfg['name']:32}  recall@{K}={r['recall@%d'%K]:.2f}  mrr={r['mrr']:.3f}")


c:\Users\rahul\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



done  v1 (250-word) · E5-base           recall@5=0.92  mrr=0.831
skip  v2 (sentence-aware) · E5-base     (missing vector_store/embeddings_v2_e5.npy — build it first)
skip  v1 (250-word) · BGE-m3            (missing vector_store/embeddings_v1_bge.npy — build it first)


## Results table (the deliverable for the professor)

In [4]:
import pandas as pd

if results:
    df = pd.DataFrame(results).sort_values("recall@%d" % K, ascending=False).reset_index(drop=True)
    display(df)
    winner = df.iloc[0]
    print(f"\nWinner so far: {winner['config']}  (recall@{K}={winner['recall@%d'%K]:.2f}, mrr={winner['mrr']:.3f})")
else:
    print("No configs scored yet — build the embeddings (last cell), then re-run.")


,config,recall@5,mrr,n_chunks
0,v1 (250-word) · E5-base,0.916667,0.831349,1307



Winner so far: v1 (250-word) · E5-base  (recall@5=0.92, mrr=0.831)


## Per-question breakdown (see *where* a config fails)

In [5]:
if per_config_rows:
    name = next(iter(per_config_rows))
    print(f"Per-question detail for: {name}\n")
    display(pd.DataFrame(per_config_rows[name]))
    misses = [r["id"] for r in per_config_rows[name] if not r["hit@%d" % K]]
    print("Missed questions:", misses or "none")
    print("(A miss can mean bad retrieval OR a bad gold question — inspect both.)")


Per-question detail for: v1 (250-word) · E5-base



,id,hit@5,first_rank
0,q01,True,1
1,q02,True,3
2,q03,True,1
3,q04,True,1
4,q05,True,1
5,q06,True,1
6,q07,True,1
7,q08,True,1
8,q09,True,1
9,q10,True,1


Missed questions: ['q12']
(A miss can mean bad retrieval OR a bad gold question — inspect both.)


## Helper — build the missing embeddings for a new config

Run this to create the `.npy` files referenced in `CONFIGS` (e.g. the
sentence-aware chunks with E5, or BGE-m3). It embeds a chunk file with the
given model, then re-run the comparison above.

`passage:` prefix is applied for e5 models only; bge-m3 gets raw text.


In [6]:
def build_embeddings(chunks_rel_path, model_name, out_rel_path):
    chunks = load_chunks(chunks_rel_path)
    model = get_model(model_name)
    is_e5 = "e5" in model_name.lower()
    prefix = "passage: " if is_e5 else ""
    texts = [prefix + c["text"] for c in chunks]
    emb = model.encode(texts, batch_size=32, normalize_embeddings=True,
                       show_progress_bar=True).astype("float32")
    np.save(BASE / out_rel_path, emb)
    print(f"Saved {emb.shape} -> {out_rel_path}")


# Uncomment the ones you want to build, then re-run the comparison cell:
# build_embeddings("vector_store/chunks_v2.jsonl", "intfloat/multilingual-e5-base", "vector_store/embeddings_v2_e5.npy")
# build_embeddings("vector_store/chunks.jsonl",    "BAAI/bge-m3",                    "vector_store/embeddings_v1_bge.npy")


## Retrieval-method comparison (fixed corpus + embeddings)

Now hold the corpus **and** embeddings fixed (the first available config) so the
*only* variable is the **retrieval method**:
- **dense** — cosine over the embeddings (what you have now)
- **BM25** — classic lexical/keyword retrieval (no embeddings)
- **hybrid** — dense + BM25 fused with Reciprocal Rank Fusion (RRF)
- **dense + reranker** — dense top-30, then re-scored by a cross-encoder
  (`BAAI/bge-reranker-v2-m3`)

Extra deps: `pip install rank-bm25` (BM25) and `sentence-transformers` (reranker).


In [7]:
# fix the base config (first one whose embeddings exist) -> only the method varies
base = next((c for c in CONFIGS if (BASE / c["emb"]).exists()), None)
assert base is not None, "No config with embeddings found — build one first."
print("Base config for retrieval comparison:", base["name"])

base_chunks = load_chunks(base["chunks"])
base_texts = [c["text"] for c in base_chunks]
base_emb = np.load(BASE / base["emb"])
base_model = get_model(base["model"])
base_prefix = base["query_prefix"]

import re
def _tok(s):
    return re.findall(r"[a-zA-ZäöüÄÖÜß0-9]+", s.lower())

from rank_bm25 import BM25Okapi
bm25 = BM25Okapi([_tok(t) for t in base_texts])
print("BM25 index built over", len(base_texts), "chunks")


Base config for retrieval comparison: v1 (250-word) · E5-base
BM25 index built over 1307 chunks


In [8]:
from collections import defaultdict

def _dense_scores(question):
    q = base_model.encode(base_prefix + question, normalize_embeddings=True).astype("float32")
    return base_emb @ q

def dense_rank(question, depth):
    return list(np.argsort(-_dense_scores(question))[:depth])

def bm25_rank(question, depth):
    return list(np.argsort(-bm25.get_scores(_tok(question)))[:depth])

def hybrid_rank(question, depth, cap=50, rrf_k=60):
    d = list(np.argsort(-_dense_scores(question))[:cap])
    b = list(np.argsort(-bm25.get_scores(_tok(question)))[:cap])
    fused = defaultdict(float)
    for rank, idx in enumerate(d):
        fused[int(idx)] += 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(b):
        fused[int(idx)] += 1.0 / (rrf_k + rank)
    return sorted(fused, key=fused.get, reverse=True)[:depth]

def score_ranker(rank_fn, gold, k=5, depth=20):
    """Generic scorer: any function question -> ranked chunk indices."""
    hits, rr, rows = 0, 0.0, []
    for g in gold:
        order = rank_fn(g["question"], depth)
        rank = None
        for pos, idx in enumerate(order, 1):
            if is_relevant(base_texts[idx], g):
                rank = pos
                break
        hit = rank is not None and rank <= k
        hits += int(hit)
        rr += (1.0 / rank) if rank else 0.0
        rows.append({"id": g["id"], "hit@%d" % k: hit, "first_rank": rank})
    n = len(gold)
    return {"recall@%d" % k: hits / n, "mrr": rr / n, "rows": rows}


In [9]:
# Reranker: dense top-N, then re-score each candidate with a cross-encoder.
RERANK_N = 30
_reranker = None

def rerank_rank(question, depth):
    global _reranker
    if _reranker is None:
        from sentence_transformers import CrossEncoder
        _reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")
    cand = dense_rank(question, RERANK_N)
    scores = _reranker.predict([[question, base_texts[i]] for i in cand])
    return [cand[j] for j in np.argsort(-np.asarray(scores))][:depth]


In [ ]:
methods = {
    "dense (cosine)": dense_rank,
    "BM25 (lexical)": bm25_rank,
    "hybrid (RRF)": hybrid_rank,
    "dense + reranker": rerank_rank,
}

retr_results = []
for name, fn in methods.items():
    try:
        r = score_ranker(fn, gold, k=K)
        retr_results.append({"method": name, "recall@%d" % K: r["recall@%d" % K], "mrr": r["mrr"]})
        print(f"done  {name:20}  recall@{K}={r['recall@%d'%K]:.2f}  mrr={r['mrr']:.3f}")
    except Exception as e:
        print(f"skip  {name:20}  ({type(e).__name__}: {e})")

import pandas as pd
rdf = pd.DataFrame(retr_results).sort_values("recall@%d" % K, ascending=False).reset_index(drop=True)
display(rdf)
if len(rdf):
    print("\nBest retrieval method:", rdf.iloc[0]["method"],
          f"(recall@{K}={rdf.iloc[0]['recall@%d'%K]:.2f}, mrr={rdf.iloc[0]['mrr']:.3f})")


done  dense (cosine)        recall@5=0.92  mrr=0.831
done  BM25 (lexical)        recall@5=0.83  mrr=0.784
done  hybrid (RRF)          recall@5=1.00  mrr=0.799


## How this covers the professor's comparisons

- **Chunking:** compare the `v1 (250-word)` vs `v2 (sentence-aware)` rows (same
  model) → the higher Recall@5 wins.
- **Embedding:** compare `E5-base` vs `BGE-m3` rows (same chunks) → winner by
  Recall@5 / MRR.
- **Retrieval:** the retrieval-method section above compares dense vs BM25 vs
  hybrid vs reranker on a fixed corpus → its own Recall@5 / MRR table.
- **Topic modeling / Generation:** scored separately (coherence for topics,
  RAGAS/LLM-judge for generation) — this harness is the retrieval half.
